# Visual Classifier — Two-Stage Fine-Tuning (Local)

This notebook performs **sequential two-stage fine-tuning** of the
`dima806/ai_vs_human_generated_image_detection` ViT model **locally**
on a MacBook Air M4 using Apple Silicon MPS acceleration.

| Stage | Dataset | Purpose |
|-------|---------|--------|
| **1** | `nebula/GenImage-arrow` (small streamed subset) | Broaden the model on diverse AI generators |
| **2** | `julienlucas/midjourney-dalle-sd-nanobananapro-dataset` | Specialise on Midjourney / DALL·E / SD |

The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

### Running Locally on Apple Silicon
1. Open this notebook in VS Code (or Jupyter).
2. Select your local Python kernel with the required packages installed.
3. Training uses the MPS backend — no CUDA required.
4. All outputs are saved inside `outputs/` relative to this notebook.

> ⚠️ **Memory:** The M4 has unified memory (16–32 GB). Batch sizes are
> kept small to avoid OOM. Adjust `BATCH_SIZE` if you have more RAM.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Configuration
Edit the values below to control dataset sizes, training hyper-parameters, etc.

In [ ]:
# ── Dataset sizes for GenImage streaming subset ──
TRAIN_PER_CLASS = 4800    # 1 000 real + 1 000 AI for training
VAL_PER_CLASS   = 550     # 250 + 250 for validation
TEST_PER_CLASS  = 1000     # 500 + 500 for test
SEED            = 42

# ── Model ──
BASE_MODEL = "dima806/ai_vs_human_generated_image_detection"

# ── Training hyper-parameters ──
EPOCHS_STAGE1   = 3
EPOCHS_STAGE2   = 3
BATCH_SIZE      = 6      # M4-friendly; increase to 8 if you have 32 GB RAM
LEARNING_RATE   = 2e-5
REPLAY_RATIO    = 0.25    # fraction of Stage-1 data mixed into Stage 2

# ── Output paths ──
STAGE1_MODEL_DIR = "outputs/models/genimage_stage1_finetuned"
STAGE2_MODEL_DIR = "outputs/models/genimage_then_julienlucas_stage2_finetuned"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup (Local – Apple Silicon)

In [ ]:
import os, sys, torch

# ── GPU / Accelerator check ──
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
elif torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("⚠️  No GPU – training will be slow on CPU")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")
print(f"PyTorch version: {torch.__version__}")

---
## 2 · Authentication

GenImage-arrow may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.

In [ ]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

---
## 3 · Imports & Suppress Warnings

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from two_stage_finetuning import (
    get_device,
    build_genimage_subset,
    load_julienlucas_dataset,
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

---
## 4 · Load the Base Model & Processor

In [ ]:
print(f"Loading base model: {BASE_MODEL}")
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
model     = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, ignore_mismatched_sizes=True
).to(DEVICE)
print(f"✅ Model loaded  –  {sum(p.numel() for p in model.parameters()):,} parameters")

---
## 5 · Stage 1 – GenImage Subset

### 5a · Build a balanced subset via streaming

GenImage-arrow is **very large**. We stream it and collect only the
samples we need, so nothing is downloaded in full.

In [ ]:
genimage_ds = build_genimage_subset(
    train_per_class=TRAIN_PER_CLASS,
    val_per_class=VAL_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
genimage_ds

### 5b · Inspect a few samples
Verify the label mapping is correct.

In [ ]:
# Show 5 examples so we can confirm the label extraction logic
for i in range(min(5, len(genimage_ds["train"]))):
    ex = genimage_ds["train"][i]
    tag = "REAL" if ex["label"] == 0 else "AI"
    print(f"  [{tag}] label={ex['label']}  path={ex.get('image_path','N/A')[:80]}")

### 5c · Fine-tune on GenImage subset

In [ ]:
model, trainer_s1 = run_training_stage(
    model=model,
    processor=processor,
    train_ds=genimage_ds["train"],
    eval_ds=genimage_ds["validation"],
    output_dir=STAGE1_MODEL_DIR,
    epochs=5,               # bumped from 3 — early stopping will cut it short if needed
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name="Stage 1 – GenImage subset",
    fp16=USE_FP16,
    freeze_layers=10,       # 🆕 freeze first 10/12 ViT layers
    augment=True,           # 🆕 random flips, JPEG jitter, colour jitter
    weight_decay=0.01,      # 🆕 L2 regularisation
    early_stopping_patience=2,  # 🆕 stop if val accuracy doesn't improve for 2 epochs
)

### 5d · Evaluate Stage 1 model on GenImage test split

In [ ]:
s1_metrics = evaluate_model(
    model=model,
    processor=processor,
    test_ds=genimage_ds["test"],
    output_prefix="genimage_stage1",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

---
## 6 · Stage 2 – julienlucas Dataset

We **continue** fine-tuning the Stage 1 model (already updated in memory).

### 6a · Load the julienlucas dataset

In [ ]:
julienlucas_ds = load_julienlucas_dataset(
    val_ratio=0.1,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
julienlucas_ds

### 6b · Fine-tune (continued) on julienlucas

In [ ]:
model, trainer_s2 = run_training_stage(
    model=model,
    processor=processor,
    train_ds=julienlucas_ds["train"],
    eval_ds=julienlucas_ds["validation"],
    output_dir=STAGE2_MODEL_DIR,
    epochs=5,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name="Stage 2 – julienlucas (with GenImage replay)",
    fp16=USE_FP16,
    replay_ds=genimage_ds["train"],
    replay_ratio=REPLAY_RATIO,
    freeze_layers=10,       # 🆕 same freezing
    augment=True,           # 🆕 augmentation
    weight_decay=0.01,      # 🆕 regularisation
    early_stopping_patience=2,  # 🆕 early stopping
)


### 6c · Evaluate final model on julienlucas test split

In [ ]:
s2_metrics = evaluate_model(
    model=model,
    processor=processor,
    test_ds=julienlucas_ds["test"],
    output_prefix="julienlucas_stage2",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

### 6d · (Optional) Re-evaluate final model on GenImage test

Check whether Stage 2 fine-tuning retained or degraded GenImage performance.

In [ ]:
s2_on_genimage = evaluate_model(
    model=model,
    processor=processor,
    test_ds=genimage_ds["test"],
    output_prefix="genimage_after_stage2",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

---
## 7 · Save Weight Delta

Instead of committing the full ~327 MB model to Git, we save only the
**weight differences** (delta) from the base model. The resulting file
is typically ~82 MB (int8 quantised) and stays under GitHub's 100 MB limit.

> This is **full fine-tuning**, not LoRA/PEFT. The delta is not a
> lightweight adapter — it encodes changes across all parameters.
> If you switch to LoRA in the future the adapter files will be much
> smaller (~5–15 MB) and can be saved separately.

In [ ]:
# Reload module to pick up any fixes
import importlib, visual_classifier
importlib.reload(visual_classifier)
from visual_classifier import save_weight_delta

delta_path, delta_mb = save_weight_delta(
    fine_tuned_model=model,
    base_model_name=BASE_MODEL,
    output_path="./fine_tuned_model_delta/weight_delta.pt",
)
print(f"\n✅ Delta saved to '{delta_path}' ({delta_mb:.2f} MB)")

---
## 8 · Summary

### What was saved

| Artefact | Path | Size |
|----------|------|------|
| Stage 1 full model | `outputs/models/genimage_stage1_finetuned/` | ~327 MB (gitignored) |
| Stage 2 full model | `outputs/models/genimage_then_julienlucas_stage2_finetuned/` | ~327 MB (gitignored) |
| Weight delta (int8) | `fine_tuned_model_delta/weight_delta.pt` | ~82 MB (tracked) |
| Stage 1 eval metrics | `outputs/genimage_stage1_eval_results.json` | tiny |
| Stage 2 eval metrics | `outputs/julienlucas_stage2_eval_results.json` | tiny |
| Post-S2 GenImage eval | `outputs/genimage_after_stage2_eval_results.json` | tiny |
| Confusion matrices | `outputs/*.png` | small |

### Next steps
- Run **`visual_classifier_eval.ipynb`** to compare original vs fine-tuned model.
- Commit the weight delta and evaluation artefacts to Git.